# Dataset preparation for the NMR structure-elucidation benchmark

This notebook documents how the benchmark set of **105 molecules** used in
*"Can AI Help Organic Chemists To Solve NMR?"* was assembled from the
**OdanChem** spectral database, and it gives the exact rules used to sort
candidate molecules into **21 functional-group classes**.

The pipeline has five stages:

1. **Source database** — start from the OdanChem collection of experimental NMR spectra.
2. **Spectral curation** — keep one high-quality `1H`/`13C` spectrum pair per molecule.
3. **Molecular complexity** — score every surviving molecule with a machine-learned complexity metric.
4. **Functional-group classification** — assign each molecule to one or more of 21 classes.
5. **Complexity-quintile sampling** — draw 5 molecules per class spanning the full difficulty range → 105 molecules.

**Scope of this notebook.** The raw OdanChem export is not redistributed with
this repository, so the batch/DataFrame machinery that streams millions of
records is omitted. What is reproduced in full here is the part that defines the
science: the class-definition rules (stage 4), which run on any single
molecule without the source dataset.


## 1. Source database: OdanChem

The starting pool of molecules is drawn from **OdanChem**, an open spectral
database of more than **20 million experimental NMR spectra** covering a broad,
chemically diverse range of organic compounds. Because it is built from real
literature measurements rather than simulated data, each spectrum is tied to a
concrete published structure — the property we exploit to guarantee that every
benchmark entry is a genuine structure/spectrum pair.


## 2. Spectral curation

Before any molecule is considered for the benchmark, the database is filtered so
that every molecule is represented by a single, trustworthy `1H`/`13C` pair.
For each molecule we keep **only the best available spectra**, both spectra were reported in the same publication.

This is what makes the pair physically consistent: the `1H` and `13C`
spectra then describe the same sample measured under the same conditions, so the
structure and its two spectra form one real experimental record.


## 3. Molecular complexity

For every molecule that survives curation we compute a **molecular complexity
score** using the machine-learned complexity metric of Ananikov and co-workers
([Chem. Sci. 2025, 16 (16), 6895–6908](https://doi.org/10.1039/D4SC07320G)). Unlike a raw atom or ring count, this
metric was trained on a large set of pairwise judgements by practising chemists,
so a single number reflects how complex a structure *feels* to a chemist. This
score is what we later grade on, quintile by quintile, inside every class.


## 4. Functional-group classification into 21 classes

Each curated molecule is assigned to one or more of 21 functional-group
classes. The membership rule is deliberately permissive:

> A molecule joins a class as soon as it contains **at least one** characteristic
> group of that class. It does **not** matter whether the molecule also carries
> other groups that place it in additional classes — a single molecule can (and
> often does) belong to several classes at once.

The rest of this section gives the exact definitions. Most classes are expressed
as one or more SMARTS substructure queries; four classes
(alkanes/haloalkanes, organophosphorus compounds, spiro compounds and
non-aromatic heterocycles) need a little dedicated logic on top of the connection
table.


### 4.1 The 21 classes at a glance

| # | Class (key) | Defining feature |
|---|-------------|------------------|
| 1 | `alkanes_haloalkanes` | only C/H/halogen atoms and fully saturated (DBE = 0) |
| 2 | `alkenes` | a non-aromatic C=C double bond |
| 3 | `alkynes` | a C#C triple bond |
| 4 | `polyarenes` | a fused-aromatic (naphthalene) core |
| 5 | `heteroarenes` | an aromatic heterocycle (pyridine, pyrrole, furan, thiophene, imidazole, triazoles) |
| 6 | `alcohols` | -OH on an sp3 carbon |
| 7 | `phenols` | -OH on an aromatic carbon |
| 8 | `ethers` | C-O-C with neither neighbour a carbonyl carbon |
| 9 | `aldehydes` | a carbonyl carbon bearing one H |
| 10 | `ketones` | C-C(=O)-C |
| 11 | `acids` | carboxylic acid / carboxylate |
| 12 | `esters` | C(=O)-O-C |
| 13 | `amides` | C(=O)-N (incl. aromatic lactams) |
| 14 | `amines` | aliphatic amine (N on sp3 C; anilines/amides/imines excluded) |
| 15 | `small_rings` | an atom in a 3- or 4-membered ring |
| 16 | `nitriles` | C#N |
| 17 | `azides` | -N=N+=N- |
| 18 | `organophosphorus` | any phosphorus atom |
| 19 | `sulfur_so` | C-S=O (sulfoxide / sulfone) |
| 20 | `spiro` | two rings sharing exactly one atom and no bond |
| 21 | `heterocycles` | a non-aromatic ring containing a heteroatom |


### 4.2 Setup

In [1]:
import re
from indigo import Indigo

# One Indigo session; stereochemistry warnings are irrelevant for classification.
IND = Indigo()
IND.setOption("ignore-stereochemistry-errors", True)

### 4.3 Class keys and human-readable names

The canonical order below (1..21) is the order in which classes are reported
throughout the study.

In [2]:
CLASS_NAMES = {
    "alkanes_haloalkanes": "1. Alkanes and haloalkanes",
    "alkenes":             "2. Alkenes",
    "alkynes":             "3. Alkynes",
    "polyarenes":          "4. Polyarenes (naphthalene)",
    "heteroarenes":        "5. Heteroarenes",
    "alcohols":            "6. Alcohols",
    "phenols":             "7. Phenols",
    "ethers":              "8. Ethers",
    "aldehydes":           "9. Aldehydes",
    "ketones":             "10. Ketones",
    "acids":               "11. Carboxylic acids",
    "esters":              "12. Esters",
    "amides":              "13. Amides",
    "amines":              "14. Amines",
    "small_rings":         "15. Small rings (3-, 4-membered)",
    "nitriles":            "16. Nitriles",
    "azides":              "17. Azides",
    "organophosphorus":    "18. Organophosphorus",
    "sulfur_so":           "19. Sulfur-containing (C-S=O)",
    "spiro":               "20. Spiro compounds",
    "heterocycles":        "21. Non-aromatic heterocycles",
}
CLASS_ORDER = list(CLASS_NAMES.keys())

### 4.4 SMARTS-based class definitions

Seventeen of the classes are defined purely by substructure. A class matches if
**any** of its SMARTS patterns is found in the molecule. The patterns are
compiled once against the Indigo session.

In [3]:
SMARTS = {
    # 2. Alkenes: any non-aromatic C=C (upper-case C = aliphatic carbon).
    "alkenes": ["C=C"],

    # 3. Alkynes: a C#C triple bond.
    "alkynes": ["C#C"],

    # 4. Polyarenes: the naphthalene substructure.
    "polyarenes": ["c1ccc2ccccc2c1"],

    # 5. Heteroarenes: pyridine, pyrrole, furan, thiophene, imidazole and
    #    1,2,3-/1,2,4-triazoles. The aromatic [n] catches both NH- and
    #    N-substituted rings (N-methylpyrrole, the imidazole in caffeine, ...).
    "heteroarenes": [
        "c1ccncc1",                          # pyridine  (6-ring, 1 N)
        "c1cc[n]c1",                         # pyrrole   (5-ring, any aromatic N)
        "c1ccoc1",                           # furan     (5-ring, O)
        "c1ccsc1",                           # thiophene (5-ring, S)
        "c1c[n]cn1",                         # imidazole (5-ring, 2 aromatic N)
        "[c;r5]1[c;r5][n;r5][n;r5][n;r5]1",  # 1,2,3-triazole
        "[c;r5]1[n;r5][c;r5][n;r5][n;r5]1",  # 1,2,4-triazole
    ],

    # 6. Alcohols: -OH on an sp3 carbon (not a phenol, not an acid).
    "alcohols": ["[CX4][OX2H1]"],

    # 7. Phenols: -OH on an aromatic carbon.
    "phenols": ["[c][OX2H1]"],

    # 8. Ethers: C-O-C (aromatic C allowed) but NOT an ester linkage ->
    #    neither neighbour of the oxygen is a carbonyl carbon.
    "ethers": ["[#6;!$([CX3]=[OX1])][OX2H0][#6;!$([CX3]=[OX1])]"],

    # 9. Aldehydes: a carbonyl carbon bearing exactly one H.
    "aldehydes": ["[CX3H1]=[OX1]"],

    # 10. Ketones: C-C(=O)-C (two carbon substituents on the carbonyl).
    "ketones": ["[#6][CX3](=[OX1])[#6]"],

    # 11. Acids: carboxylic acid / carboxylate.
    "acids": ["[CX3](=[OX1])[OX1H0-,OX2H1]"],

    # 12. Esters: C(=O)-O-C.
    "esters": ["[#6][CX3](=[OX1])[OX2H0][#6]"],

    # 13. Amides: C(=O)-N. Element-level atoms ([#6]/[#7]) also catch aromatic
    #     lactams (uracil, the xanthine rings of caffeine, ...).
    "amides": ["[#6X3](=[OX1])[#7X3]"],

    # 14. Amines: aliphatic amine (N on an sp3 C), excluding amides / imines /
    #     nitro / charged N. Anilines are deliberately NOT counted here.
    "amines": ["[NX3;!$([NX3][CX3]=[OX1]);!$([NX3]=*);!$([N+])][CX4]"],

    # 15. Small rings: an atom in a 3- or 4-membered ring.
    "small_rings": ["[r3,r4]"],

    # 16. Nitriles: C#N.
    "nitriles": ["[CX2]#[NX1]"],

    # 17. Azides: -N=N+=N- (both protonation drawings).
    "azides": ["[$(N=[N+]=[N-]),$([N-]=[N+]=N)]"],

    # 19. Sulfur-containing: C-S=O (sulfoxide S=O and sulfone S(=O)=O).
    "sulfur_so": ["[#6][#16]=[#8]"],
}

# Compile every pattern once against the session.
COMPILED = {key: [IND.loadSmarts(p) for p in pats] for key, pats in SMARTS.items()}

### 4.5 Classes that need dedicated logic

Four classes cannot be captured by a single substructure query and are computed
directly from the connection table:

- **`alkanes_haloalkanes`** — only C/H/halogen atoms *and* full saturation, i.e.
  a degree of unsaturation (DBE) of exactly 0.
- **`organophosphorus`** — the presence of any phosphorus atom.
- **`spiro`** — a pair of rings sharing exactly one atom and no bonds.
- **`heterocycles`** — a non-aromatic ring that contains a heteroatom (so
  aromatic heterocycles fall under `heteroarenes`, not here).

The DBE and ring helpers below are the minimal amount of bookkeeping these four
definitions require.

In [ ]:
HAL = ("F", "Cl", "Br", "I")
ALK_ELEMS = {"C", "H", "F", "Cl", "Br", "I"}


def gross_counts(mol):
    """Parse Indigo's grossFormula() ('C2 H6 O') into {element: count}."""
    counts = {}
    for tok in mol.grossFormula().split():
        m = re.match(r"([A-Z][a-z]?)(\d*)", tok)
        if not m:
            continue
        el = m.group(1)
        n = int(m.group(2)) if m.group(2) else 1
        counts[el] = counts.get(el, 0) + n
    return counts


def dbe(counts):
    """Degree of unsaturation from the gross formula: (2C + 2 + N - H - X) / 2."""
    C = counts.get("C", 0)
    H = counts.get("H", 0)
    N = counts.get("N", 0)
    X = sum(counts.get(x, 0) for x in HAL)
    return (2 * C + 2 + N - H - X) / 2.0


def is_alkane_haloalkane(counts):
    """1. Only C/H/halogen atoms and saturated (DBE == 0)."""
    if "C" not in counts:
        return False
    if any(el not in ALK_ELEMS for el in counts):
        return False
    return dbe(counts) == 0


def has_phosphorus(mol):
    """18. Any phosphorus atom."""
    return any(a.symbol() == "P" for a in mol.iterateAtoms())


def ring_sets(mol):
    """SSSR rings as (set of atom indices, set of bond indices)."""
    rings = []
    for ring in mol.iterateSSSR():
        atoms = set(a.index() for a in ring.iterateAtoms())
        bonds = set(b.index() for b in ring.iterateBonds())
        rings.append((atoms, bonds))
    return rings


def is_spiro(rings):
    """20. A pair of rings sharing exactly one atom and zero bonds."""
    n = len(rings)
    for i in range(n):
        for j in range(i + 1, n):
            shared_atoms = rings[i][0] & rings[j][0]
            shared_bonds = rings[i][1] & rings[j][1]
            if len(shared_atoms) == 1 and len(shared_bonds) == 0:
                return True
    return False


def is_heterocycle(mol):
    """21. A non-aromatic SSSR ring that contains a heteroatom (not C/H).
    Aromatic rings (Indigo bond order 4) are skipped, so heteroarenes are
    excluded."""
    for ring in mol.iterateSSSR():
        has_hetero = any(a.symbol() not in ("C", "H") for a in ring.iterateAtoms())
        if not has_hetero:
            continue
        is_aromatic = any(b.bondOrder() == 4 for b in ring.iterateBonds())
        if not is_aromatic:
            return True
    return False

### 4.6 Classifying a single molecule

`classify()` ties the pieces together. It loads a molecule (a MOLFILE block or a
SMILES string), aromatizes it once, and then evaluates every class: the SMARTS
classes through a substructure matcher and the four special classes through the
helpers above. It returns the list of matched class keys in canonical order —
possibly several, reflecting the permissive membership rule.

In [5]:
def classify(mol_input, as_names=False):
    """Return the classes a molecule belongs to (keys, or human-readable names)."""
    mol = IND.loadMolecule(mol_input)          # MOLFILE block or SMILES
    try:
        mol.aromatize()                        # needed for aromatic SMARTS + heterocycle test
    except Exception:
        pass

    counts = gross_counts(mol)
    matcher = IND.substructureMatcher(mol)
    rings = ring_sets(mol)

    flags = {k: False for k in CLASS_ORDER}
    flags["alkanes_haloalkanes"] = is_alkane_haloalkane(counts)
    for key in SMARTS:                          # SMARTS classes (2..17, 19)
        flags[key] = any(matcher.match(q) is not None for q in COMPILED[key])
    flags["organophosphorus"] = has_phosphorus(mol)
    flags["spiro"] = is_spiro(rings)
    flags["heterocycles"] = is_heterocycle(mol)

    keys = [k for k in CLASS_ORDER if flags[k]]
    return [CLASS_NAMES[k] for k in keys] if as_names else keys

## 5. Complexity-quintile sampling → 105 molecules

With every candidate molecule (a) curated to a single high-quality spectrum pair,
(b) scored for complexity, and (c) tagged with its 21-class membership, the final
benchmark is drawn class by class:

- within each of the 21 classes we split the class-wide complexity distribution
  into five quintiles, and
- draw one molecule at random from each quintile.

This yields five molecules per class, evenly spanning the class's full
difficulty range, for a total of **21 x 5 = 105 molecules**. Because one molecule
can belong to several classes, the assembled set is finally checked for repeated
structures across classes; no duplicates were found, so all 105 entries are
distinct. The resulting benchmark is stored in
`dataset/dataset_selected_clean_105.json`.

> The stage-5 sampling and the stage 1-3 streaming operate on the full OdanChem
> export, which is not shipped with this repository; they are therefore described
> rather than executed here. Everything in **stage 4** above, however, is complete
> and runs on any molecule.
